# Julia Fundamentals

## No types, no performance

Although types aren't always explicitly visible, **Julia is built around types**.

High performance codes in Julia make good use of the type system.

In [1]:
3 + 2

5

In [2]:
3 + 2.0

5.0

In [3]:
typeof(5)

Int64

In [7]:
typeof(5.0) 

Float64

In [8]:
using About

In [9]:
about(5)

Int64 (<: Signed <: Integer <: Real <: Number <: Any), occupies 8B.

 0000000000000000000000000000000000000000000000000000000000000101
 = +5


In [10]:
about(5.0)

Float64 (<: AbstractFloat <: Real <: Number <: Any), occupies 8B.

 0100000000010100000000000000000000000000000000000000000000000000 
 ╨└────┬────┘└────────────────────────┬─────────────────────────┘
 +    2^2   ×                1.250000000000000000                
 = 5.0000000000000000


**Concrete types**
* types of values ("objects")
* specify precise structure of data
* Example: `Float64` (IEEE-754 standard)

**Abstract types**
* cannot be instantiated
* useful to share behavior (methods) between multiple concrete types (their descendants)
* Examples:
    * The abstract type `Number` indicates that one can do number-like things, e.g. `+`,`-`,`*`, and `/`, with values that have `typeof(value) <: Number`. In this category we have (concrete) things like `Float64` and `Int32` numbers.

    * An `AbstractArray` is a type that indicates that we can, for instance, get elements at a particular index, e.g. `A[i]`. Examples include regular arrays (`Array`), as well as ranges (`UnitRange`).

In [11]:
Int64 <: Number

true

In [12]:
Float64 <: Number

true

In [13]:
String <: Number

false

### Inspecting the type tree

In [14]:
using AbstractTrees
AbstractTrees.children(x) = subtypes(x)

In [15]:
print_tree(Number) # print a branch of the type tree

Number
├─ MultiplicativeInverse
│  ├─ SignedMultiplicativeInverse
│  └─ UnsignedMultiplicativeInverse
├─ Complex
└─ Real
   ├─ AbstractFloat
   │  ├─ BigFloat
   │  ├─ BFloat16
   │  ├─ Float16
   │  ├─ Float32
   │  └─ Float64
   ├─ AbstractIrrational
   │  └─ Irrational
   ├─ Integer
   │  ├─ Bool
   │  ├─ Signed
   │  │  ├─ BigInt
   │  │  ├─ Int128
   │  │  ├─ Int16
   │  │  ├─ Int32
   │  │  ├─ Int64
   │  │  └─ Int8
   │  └─ Unsigned
   │     ├─ UInt128
   │     ├─ UInt16
   │     ├─ UInt32
   │     ├─ UInt64
   │     └─ UInt8
   └─ Rational


Everything is a subtype of `Any`. It's the root of the type tree.

Note that **concrete types are the leaves of the type tree** whereas **abstract types are nodes** in the type graph.

### `Array` and `UnitRange`

In [16]:
vec = [1,2,3]

3-element Vector{Int64}:
 1
 2
 3

In [17]:
typeof(vec)

Vector{Int64} (alias for Array{Int64, 1})

In [18]:
M = rand(3,3)

3×3 Matrix{Float64}:
 0.325293   0.70185   0.547059
 0.817941   0.996273  0.968762
 0.0872913  0.504802  0.716074

In [19]:
xx

LoadError: syntax: all-underscore identifiers are write-only and their values cannot be used in expressions

In [20]:
typeof(M)

Matrix{Float64} (alias for Array{Float64, 2})

`Array` is a **parametric type**. The type parameters are

* `Int64` (element type)
* `1` (dimensionality)

Hence `Array{Int64, 1}` means that we have a vector that can hold 64-bit integers.

In [ ]:
eltype(vec)

In [ ]:
typeof(vec) <: AbstractArray

In [ ]:
vec[2]

In [ ]:
length(vec)

#### `UnitRange`

In [21]:
x = 1:30

1:30

Unit range does not store the elements, it just stores the first and the last and the step

In [22]:
typeof(x)

UnitRange{Int64}

In [23]:
typeof(x) <: AbstractArray

true

Because it is a subtype of `AbstractArray` it should support common array functions.

In [24]:
x[3]

3

In [25]:
eltype(x)

Int64

However, it's not implemented like a regular `Array` at all. In fact, **it's just two numbers**, the start and stop values of the range it represents.

In [26]:
fieldnames(typeof(x))

(:start, :stop)

It thus represents a range but with a much lower memory footprint than a corresponding array.

## Functions, methods, and dispatch

Let's define a *function* that calculates the absolute value of a number (like the built-in `abs` already does).

**How would we practically calculate the absolute values of the numbers $-4.32$ and $1.0 + 1.0i$?**

* Real number
  * "Drop the sign."
    * => `myabs(-4.32) = 4.32`
* Complex number:
  * "Square root of z times the complex conjugate of z."
    * => `myabs(1.0 + 1.0im) = sqrt(2) ≈ 1.414`

We see that the *methods* that we use depend on the type of the number.

While the single **function** represents the *what* ("calculate the absolute value"), there might be different **methods** describing the *how*.

We can use the `::` operator to annotate function arguments with types and define different methods.

In [27]:
myabs(x::Float64) = sign(x) * x

myabs (generic function with 1 method)

In [28]:
myabs(-4.32)

4.32

In [29]:
myabs(1.0 + 1.0im)

LoadError: MethodError: no method matching myabs(::ComplexF64)
The function `myabs` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  myabs([91m::Float64[39m)
[0m[90m   @[39m [35mMain[39m [90m[4mIn[27]:1[24m[39m


In [31]:
myabs(z::ComplexF64) = sqrt(real(z * conj(z)))

myabs (generic function with 2 methods)

In [32]:
myabs(1.0 + 1.0im)

1.4142135623730951

In [33]:
methods(myabs)

# 2 methods for generic function "myabs" from Main:
 [1] myabs(z::ComplexF64)
     @ In[31]:1
 [2] myabs(x::Float64)
     @ In[27]:1

One can check which particular method is being used with the `@which` macro.

In [34]:
@which myabs(-4.32)

myabs(x::Float64)
     @ Main In[27]:1

In [35]:
@which myabs(1.0 + 1.0im)

myabs(z::ComplexF64)
     @ Main In[31]:1

Note that we should better loosen our type restrictions:

In [36]:
myabs(-3)

LoadError: MethodError: no method matching myabs(::Int64)
The function `myabs` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  myabs([91m::ComplexF64[39m)
[0m[90m   @[39m [35mMain[39m [90m[4mIn[31]:1[24m[39m
[0m  myabs([91m::Float64[39m)
[0m[90m   @[39m [35mMain[39m [90m[4mIn[27]:1[24m[39m


In [37]:
myabs(1 + 1im)

LoadError: MethodError: no method matching myabs(::Complex{Int64})
The function `myabs` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  myabs([91m::ComplexF64[39m)
[0m[90m   @[39m [35mMain[39m [90m[4mIn[31]:1[24m[39m
[0m  myabs([91m::Float64[39m)
[0m[90m   @[39m [35mMain[39m [90m[4mIn[27]:1[24m[39m


In [40]:
myabs(x::Real) = sign(x) * x
myabs(z::Complex) = sqrt(real(z * conj(z)))

myabs (generic function with 4 methods)

In [41]:
myabs(-3)

3

As we will understand later, type annotations in function signatures **virtually never affect performance**! One should therefore generally make them as generic as possible but as specific as necessary.

### Multiple dispatch

Which method gets executed when you call a generic function `f` for a given set of input arguments?

**Answer:** Julia always chooses the **most specific method** by considering **all input argument types**.

(Since methods belong to generic functions rather than objects no function argument is special.)

In [42]:
                              f(a, b) = "a and b are anything"
                      f(a::Number, b) = "a is a number, b is anything"
                      f(a, b::Number) = "a is anything, b is a number"
              f(a::Number, b::Number) = "a and b are both numbers"
f(a::AbstractFloat, b::AbstractFloat) = "a and b are both floats"
            f(a::Float32, b::Float32) = "a and b are both 32-bit floats"

f (generic function with 6 methods)

In [43]:
methods(f)

# 6 methods for generic function "f" from Main:
 [1] f(a::Float32, b::Float32)
     @ In[42]:6
 [2] f(a::AbstractFloat, b::AbstractFloat)
     @ In[42]:5
 [3] f(a::Number, b::Number)
     @ In[42]:4
 [4] f(a::Number, b)
     @ In[42]:2
 [5] f(a, b::Number)
     @ In[42]:3
 [6] f(a, b)
     @ In[42]:1

In [44]:
f("Hello", "Julia")

"a and b are anything"

In [47]:
f(1.5, 2)

"a and b are both numbers"

In [46]:
f(1, "Julia")

"a is a number, b is anything"

In [48]:
f(1, 2)

"a and b are both numbers"

In [49]:
f(1.2, 3.4)

"a and b are both floats"

In [50]:
f(1.2f0, 3.4f0) # 1.2f0 is the literal way to write Float32(1.2)

"a and b are both 32-bit floats"

In [51]:
@which f(1.2, 3.4)

f(a::AbstractFloat, b::AbstractFloat)
     @ Main In[42]:5

In [52]:
@which f(1, "HLRS")

f(a::Number, b)
     @ Main In[42]:2

### Built-in Julia function

**Julia's built-in functions are not special by any means.**

In [53]:
methods(+)

# 190 methods for generic function "+" from Base:
   [1] +(x::Bool, z::Complex{Bool})
     @ complex.jl:301
   [2] +(x::Bool, y::Bool)
     @ bool.jl:169
   [3] +(x::Bool)
     @ bool.jl:166
   [4] +(x::Bool, z::Complex)
     @ complex.jl:308
   [5] +(x::Bool, y::T) where T<:AbstractFloat
     @ bool.jl:176
   [6] +(a::BigFloat, b::BigFloat, c::BigFloat, d::BigFloat, e::BigFloat)
     @ mpfr.jl:765
   [7] +(x::BigFloat, y::BigFloat)
     @ mpfr.jl:578
   [8] +(a::BigFloat, b::BigFloat, c::BigFloat)
     @ mpfr.jl:752
   [9] +(a::BigFloat, b::BigFloat, c::BigFloat, d::BigFloat)
     @ mpfr.jl:758
  [10] +(x::BigFloat, c::BigInt)
     @ mpfr.jl:609
  [11] +(x::BigFloat, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ mpfr.jl:585
  [12] +(x::BigFloat, c::Union{Int16, Int32, Int64, Int8})
     @ mpfr.jl:593
  [13] +(x::BigFloat, c::Union{Float16, Float32, Float64})
     @ mpfr.jl:601
  [14] +(dt::Dates.DateTime, y::Dates.Year)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:25
  [15] +(dt::Dates.DateTime, z::Dates.Month)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:49
  [16] +(x::Dates.DateTime, y::Dates.Quarter)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:77
  [17] +(x::Dates.DateTime, y::Dates.Period)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:83
  [18] +(level::Base.CoreLogging.LogLevel, inc::Integer)
     @ logging/logging.jl:134
  [19] +(t::Dates.Time, dt::Dates.Date)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:22
  [20] +(x::Dates.Time, y::Dates.TimePeriod)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:85
  [21] +(x::Rational{BigInt}, y::Rational{BigInt})
     @ gmp.jl:1097
  [22] +(dt::Dates.Date, t::Dates.Time)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:21
  [23] +(dt::Dates.Date, y::Dates.Year)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:29
  [24] +(dt::Dates.Date, z::Dates.Month)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:56
  [25] +(x::Dates.Date, y::Dates.Quarter)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:75
  [26] +(x::Dates.Date, y::Dates.Week)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:79
  [27] +(x::Dates.Date, y::Dates.Day)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/arithmetic.jl:81
  [28] +(B::BitMatrix, J::LinearAlgebra.UniformScaling)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/LinearAlgebra/src/uniformscaling.jl:154
  [29] +(c::BigInt, x::BigFloat)
     @ mpfr.jl:614
  [30] +(x::BigInt, y::BigInt)
     @ gmp.jl:501
  [31] +(a::BigInt, b::BigInt, c::BigInt)
     @ gmp.jl:541
  [32] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt)
     @ gmp.jl:542
  [33] +(a::BigInt, b::BigInt, c::BigInt, d::BigInt, e::BigInt)
     @ gmp.jl:543
  [34] +(x::BigInt, y::BigInt, rest::BigInt...)
     @ gmp.jl:687
  [35] +(x::BigInt, c::Union{UInt16, UInt32, UInt64, UInt8})
     @ gmp.jl:549
  [36] +(x::BigInt, c::Union{Int16, Int32, Int64, Int8})
     @ gmp.jl:555
  [37] +(x::Dates.CompoundPeriod, y::Dates.CompoundPeriod)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/periods.jl:335
  [38] +(x::Dates.CompoundPeriod, y::Dates.Period)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/periods.jl:333
  [39] +(x::Dates.CompoundPeriod, y::Dates.TimeType)
     @ /mnt/raid/soft/julia/julia-1.13.0/share/julia/stdlib/v1.13/Dates/src/periods.jl:363
  [40] +(z::Complex{Bool}, x::Bool)
     @ complex.jl:302
  [41] +(z::Complex{Bool}, x::Real)
     @ complex.jl:316
  [42] +(::Missing, ::Missing)
     @ missing.jl:119
  [43] +(::Missing)
     @ m

In [54]:
@which 1//2 + 1//4

+(x::Rational, y::Rational)
     @ Base rational.jl:356

We can easily modify or add methods to them as well.

In [55]:
# concrete types that hold no data (they only exist for dispatch)
struct Zero end
struct NonZero end

import Base: + # explicitly import functions to override/add methods

+(x::Zero, y::Zero) = Zero()
+(x::Zero, y::NonZero) = NonZero()
+(x::NonZero, y::NonZero) = NonZero()

+ (generic function with 193 methods)

In [56]:
Zero() + Zero()

Zero()

In [57]:
Zero() + NonZero()

NonZero()

### Type parameters in function signatures (if time permits)

Naive approach:

In [ ]:
myfunc(v::Vector{Real}) = "I'm a real vector!"

In [ ]:
myfunc([1.0,2.0,3.0])

Huh? What's going on?

Note that although we have

In [ ]:
Float64 <: Real

parametric types have the following - at first somewhat counterintuitive - property

In [ ]:
Vector{Float64} <: Vector{Real}

In [ ]:
[1.0,2.0,3.0] isa Vector{Real}

The crucial point is that `Vector{Real}` describes a concrete **heterogeneous** vector: Each element can be of any type `T <: Real`.

In [ ]:
Real[1, 2.2, 13f0]

(Because it is a concrete type - a leaf of the type tree - it immediately follows that there are no subtypes.)

What we often actually *mean* when writing `myfunc(v::Vector{Real}) = ...` is

In [ ]:
myfunc(v::Vector{T}) where T<:Real = "I'm a real vector!"

In [ ]:
myfunc([1.0,2.0,3.0])

Here, `Vector{T} where T <: Real` describes all **homogeneous** vector types: Each vector can only hold elements of a specific type `T` that is a subtype of `Real`. This includes, for example: `Vector{Float64}, Vector{Int64}, Vector{Int32}, Vector{AbstractFloat}, ...`

(Side comment: We can also use the `where` notation to write out our naive `Vector{Real}` from above in a more explicit way: `Vector{Real} === Vector{T where T<:Real}`. Note that the "degree of freedom", `T`, is inside of the curly braces.)

# Core messages of this notebook

* **Concrete types** describe data structures, i.e. concrete implementations.
* **Abstract types** are nodes in the type tree and can be used to share "behavior" (i.e. methods).
* A **function** (the what) can have multiple **methods** (the how).
* Types in function signatures serve as filters. **Avoid writing overly-specific types**.
* **Multiple dispatch**: Julia selects the method to run based on the types of all input arguments and chooses the most specialized one.
* Types can have parameters, i.e. `Vector{Float64}`. We can use the notation `Vector{T} where T<:SomeSuperType` to describe sets of restriced *homogeneous* vectors.